
# Set E — From Passive to Spiking, then Adaptation/Bursting (LEGO–Colab)

**Final (Option 1 style)** — 2026-03-09  \
**Author:** Satish Nair

Run the **Starter** once, then execute **E1–E12** in order. Each section includes a short **systems‑theoretic callout** and a **Try with Copilot** prompt.

---
## Table of Contents
- [🔰 Starter](#starter)
- [E1 — Add HH and find threshold](#e1)
- [E2 — Dissect Na/K roles](#e2)
- [E3 — Refractory (paired pulses)](#e3)
- [E4 — f–I curve and rheobase](#e4)
- [E5 — Waveform landmarks](#e5)
- [E6 — Propagation teaser (axon chain)](#e6)
- [E7 — Spike‑frequency adaptation (didactic surrogate)](#e7)
- [E8 — Quantify adaptation](#e8)
- [E9 — Bursting‑like bouts (drive envelope)](#e9)
- [E10 — AHP/ADP notes](#e10)
- [E11 — Phase‑plane glimpse](#e11)
- [E12 — Numerical tips](#e12)
- [Reflection](#reflection)
---


### 🔰 Starter (run once) <a id='starter'></a>

In [ ]:

# --- Set E Starter: NEURON + baseline cell (passive -> HH) ---
!pip -q install neuron==8.2.4
try:
    from neuron import h, gui
except Exception:
    from neuron import h
from neuron.units import ms, mV
import matplotlib.pyplot as plt
import numpy as np

h.load_file("stdrun.hoc")

# Single compartment soma
soma = h.Section(name='soma')
soma.L = 20; soma.diam = 20; soma.Ra = 100
soma.insert('pas'); soma.g_pas = 0.0002; soma.e_pas = -65; soma.cm = 1.0

# Recording
t = h.Vector().record(h._ref_t)
v = h.Vector().record(soma(0.5)._ref_v)

def run(sim_ms=60, v_init=-65, title=""):
    h.finitialize(v_init * mV)
    h.continuerun(sim_ms * ms)
    plt.figure(figsize=(7,3.3))
    plt.plot(t, v, 'k')
    plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.title(title)
    plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()

print("Set E Starter ready.")


## E1 — Add HH channels and search for threshold <a id='e1'></a>

In [ ]:

soma.insert('hh')
stim = h.IClamp(soma(0.5)); stim.delay=5; stim.dur=20
for A in [0.03, 0.05, 0.07, 0.09]:
    stim.amp = A
    run(sim_ms=35, title=f"E1: I={A} nA (threshold search)")


> **Systems callout — Emergent threshold**  
- With HH, state expands to gating variables (m,h,n).
- Positive feedback in Na⁺ activation competes with K⁺ activation to create a threshold.

**Try with Copilot**  
> Suggest an amplitude just below and just above threshold and explain using Na/K feedback timing.

## E2 — Dissect channel roles by toggling gNa/gK <a id='e2'></a>

In [ ]:

# Standard hh params
soma.gnabar_hh = 0.12; soma.gkbar_hh = 0.036
stim.amp = 0.08
run(sim_ms=35, title="E2: Normal HH")

soma.gkbar_hh = 0.0
run(sim_ms=35, title="E2: No K conductance (repolarization impaired)")

soma.gkbar_hh = 0.036; soma.gnabar_hh = 0.0
run(sim_ms=35, title="E2: No Na conductance (no upstroke)")

# Restore
soma.gnabar_hh = 0.12; soma.gkbar_hh = 0.036


> **Systems callout — Functional decomposition**  
- Na⁺ conductance mediates rapid upstroke; K⁺ conductance mediates repolarization and AHP.
- Turning off one reveals each role and its effect on spike shape.

**Try with Copilot**  
> Predict how a 20% decrease in gNa shifts threshold and max dV/dt.

## E3 — Refractory periods (paired pulses) <a id='e3'></a>

In [ ]:

stim1 = h.IClamp(soma(0.5)); stim1.delay=5;   stim1.dur=2; stim1.amp=0.2
stim2 = h.IClamp(soma(0.5)); stim2.dur=2;     stim2.amp=0.2
for isi in [2, 4, 6, 8, 10, 12]:
    stim2.delay = 5 + isi
    run(sim_ms=22, title=f"E3: Paired pulses, ISI={isi} ms")


> **Systems callout — Recovery of excitability**  
- Absolute refractory from Na⁺ inactivation; relative refractory from residual K⁺ activation.
- Spike probability and latency recover as ISI increases.

**Try with Copilot**  
> Estimate the relative refractory period and relate it to the K⁺ tail.

## E4 — f–I (rate–current) curve and rheobase <a id='e4'></a>

In [ ]:

amps = np.linspace(0.04, 0.15, 8)
stim = h.IClamp(soma(0.5)); stim.delay=5; stim.dur=300
rates = []
for A in amps:
    stim.amp = A
    h.finitialize(-65*mV); h.continuerun(320*ms)
    vv = np.array(v); tt = np.array(t)
    crossings = (vv[1:]>=0) & (vv[:-1]<0)
    spikes = np.sum(crossings)
    rate = 1000*spikes/300.0
    rates.append(rate)
import matplotlib.pyplot as plt
plt.figure(figsize=(5.4,3.2))
plt.plot(amps, rates, 'o-'); plt.xlabel('Current (nA)'); plt.ylabel('Rate (Hz)')
plt.title('E4: f–I (rough)'); plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()


> **Systems callout — From excitability to rate**  
- Rheobase marks minimal steady current to spike; f–I slope reflects gain of the spiking mechanism.
- Nonlinearity near rheobase and saturation at high drive are expected.

**Try with Copilot**  
> Fit a line to the mid‑range of your f–I and report slope/intercept; discuss deviations near rheobase.

## E5 — Waveform landmarks (Vmax, dV/dt, AHP) <a id='e5'></a>

In [ ]:

stim = h.IClamp(soma(0.5)); stim.amp=0.1; stim.delay=5; stim.dur=20
h.finitialize(-65*mV); h.continuerun(35*ms)
vv = np.array(v); tt = np.array(t)
dvdtt = np.gradient(vv, tt)
print("Peak Vm ~", vv.max(), "mV; max dV/dt ~", dvdtt.max(), "mV/ms")


> **Systems callout — Landmarks for comparison**  
- Vmax and max dV/dt summarize Na⁺ activation strength/speed; AHP depth reflects K⁺ activation.
- Useful for comparing to experimental spikes and for parameter tuning.

**Try with Copilot**  
> Reduce gNa by 10% and report changes in Vmax and max dV/dt.

## E6 — Propagation teaser (short HH axon chain) <a id='e6'></a>

In [ ]:

axon = [h.Section(name=f'ax_{i}') for i in range(5)]
for i,sec in enumerate(axon):
    sec.L = 100; sec.diam = 1.0; sec.Ra=100
    sec.insert('hh')
    if i>0: sec.connect(axon[i-1](1.0))
stim_ax = h.IClamp(axon[0](0.5)); stim_ax.delay=1; stim_ax.dur=1; stim_ax.amp=0.4
recs = [h.Vector().record(sec(0.5)._ref_v) for sec in axon]; t_ax = h.Vector().record(h._ref_t)

h.finitialize(-65*mV); h.continuerun(8*ms)
plt.figure(figsize=(7,3.3))
for i,vec in enumerate(recs): plt.plot(t_ax, vec, label=f'ax_{i}')
plt.legend(); plt.xlabel('Time (ms)'); plt.ylabel('mV'); plt.title('E6: Propagation teaser'); plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()


> **Systems callout — Traveling spikes**  
- Adjacent excitable compartments regenerate the spike; velocity depends on geometry and channel densities.
- Sets the stage for myelination effects in other sets.

**Try with Copilot**  
> Estimate propagation velocity from the time to peak across nodes.

## E7 — Spike‑frequency adaptation (didactic surrogate) <a id='e7'></a>

In [ ]:

soma.insert('hh')
stim = h.IClamp(soma(0.5)); stim.delay=5; stim.dur=1000; stim.amp=0.12
synI = h.AlphaSynapse(soma(0.5))
synI.onset = 100; synI.tau = 200; synI.gmax = 0.002; synI.e = -70
h.finitialize(-65*mV); h.continuerun(1100*ms)
plt.figure(figsize=(7,3.3)); plt.plot(t, v, 'k')
plt.xlabel('Time (ms)'); plt.ylabel('mV'); plt.title('E7: Didactic SFA surrogate')
plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()


> **Systems callout — Simple SFA surrogate**  
- A slow hyperpolarizing conductance (here via AlphaSynapse) reduces firing over time.
- Captures *qualitative* adaptation without adding a full slow K⁺ current model.

**Try with Copilot**  
> Double `synI.tau` and report change in adaptation time course; suggest a biophysical analog.

## E8 — Quantify adaptation (instantaneous rate) <a id='e8'></a>

In [ ]:

vv = np.array(v); tt = np.array(t)
cross = np.where((vv[1:]>=0) & (vv[:-1]<0))[0]
spk_t = tt[cross]
isis = np.diff(spk_t)
inst_rate = 1000.0/isis
plt.figure(figsize=(6,3.1))
plt.plot(spk_t[1:], inst_rate, 'o-')
plt.xlabel('Time (ms)'); plt.ylabel('Inst. rate (Hz)'); plt.title('E8: Adaptation quantification')
plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()
if len(inst_rate)>4:
    early = inst_rate[:3].mean(); late = inst_rate[-3:].mean()
    print(f"Adaptation ratio (late/early) ≈ {late/early:.2f}")


> **Systems callout — From spikes to metrics**  
- Instantaneous rate 1/ISI reveals adaptation trajectory; ratios provide compact summaries.
- Useful for comparing conditions or parameter sets.

**Try with Copilot**  
> Compute adaptation ratio across multiple `gmax` values and plot vs `gmax`.

## E9 — Bursting‑like bouts (slow drive envelope stand‑in) <a id='e9'></a>

In [ ]:

ic = h.IClamp(soma(0.5)); ic.delay=0; ic.dur=1e9; ic.amp=0.0
T = 4000
tt = np.arange(0, T, 0.1)
envelope = 0.07 + 0.05*np.sin(2*np.pi*tt/1500.0)
ivec = h.Vector(envelope); tvec = h.Vector(tt)
ivec.play(ic._ref_amp, tvec, 1)
h.finitialize(-65*mV); h.continuerun(T*ms)
plt.figure(figsize=(7,3.3)); plt.plot(t, v, 'k')
plt.xlabel('Time (ms)'); plt.ylabel('mV'); plt.title('E9: Bursting-like bouts (drive envelope)')
plt.grid(True, alpha=0.25); plt.tight_layout(); plt.show()


> **Systems callout — Slow modulation → bursts**  
- Slowly varying inputs move the cell across threshold, creating burst‑like epochs.
- A stand‑in for intrinsic slow currents or modulatory synaptic input.

**Try with Copilot**  
> Change envelope frequency/amplitude to control burst duration and inter‑burst interval; explain trends.

## E10 — AHP & ADP notes (see text module) <a id='e10'></a>
*Notes placeholder:* discuss after‑hyperpolarization (AHP) and after‑depolarization (ADP) mechanisms and their signatures in traces.

## E11 — Phase‑plane (V, dV/dt) glimpse <a id='e11'></a>

In [ ]:

stim = h.IClamp(soma(0.5)); stim.delay=5; stim.dur=40; stim.amp=0.12
h.finitialize(-65*mV); h.continuerun(70*ms)
vv = np.array(v); tt = np.array(t); dv = np.gradient(vv, tt)
plt.figure(figsize=(4.4,4.1))
plt.plot(vv, dv, 'k')
plt.xlabel('V (mV)'); plt.ylabel('dV/dt (mV/ms)'); plt.title('E11: Phase-plane glimpse')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


> **Systems callout — Geometry of spiking**  
- The phase‑plane loop reveals fast upstroke and slower recovery branches.
- Helpful for contrasting with reduced models (e.g., FHN, Izhikevich).

**Try with Copilot**  
> Overlay loops at two current levels and compare shapes (onset dynamics vs recovery).

## E12 — Numerical tips (dt, stability) <a id='e12'></a>

In [ ]:

h.dt = 0.0125
print('Time step set to', h.dt, 'ms')


> **Systems callout — Integration settings**  
- Smaller dt resolves faster dynamics but increases cost.
- Check for numerical artifacts by halving dt and comparing key features (e.g., f–I slope). 

## Reflection <a id='reflection'></a>
- How do Na⁺ and K⁺ conductances jointly determine threshold, Vmax, and refractory dynamics?  
- Where does adaptation originate in your surrogate and what biophysical currents could realize it?  
- How does slow modulation induce bursting and how might intrinsic slow currents replace external envelopes?